In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import scipy.stats as stats
from matplotlib.ticker import MaxNLocator, MultipleLocator
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. Simulation Settings
# ==========================================
num_simulations = 999
c_resolution = 300

# ==========================================
# 2. Biological Parameters
# ==========================================
S0_total = 1e7
R0_total = 1e6

d = 0.07
K_cap = 1e9
Vsmin = -0.1; Vrmin = -0.1

Vsmax_range = (0.98, 1.0)
Vrmax_range = (0.89, 0.90)

# Strictly refer to the asymmetric competition global parameters of Study A/B
DELTA_MIN, DELTA_MAX = 0.02, 0.12
GAMMA_IB_CAP = 1.40
EPS = 1e-6

dt = 0.5
total_time = 72
time_steps = int(total_time / dt)

# ==========================================
# 3. Core Simulation Function
# ==========================================
def run_simulation(mode, param_ranges, c_max):
    c_range = np.linspace(0, c_max, c_resolution)
    results = {}
    print(f"--- Running Simulation: {mode} (N={num_simulations}, Stratified Sampling) ---")
    
    for idx, (r_min, r_max) in enumerate(param_ranges):
        msc_list = []
        all_curves_matrix = []
        
        print(f"   Processing Group {r_min}-{r_max}...")
        
        for sim in range(num_simulations):
            if sim % 100 == 0:
                print(f"     -> Simulating {sim}/{num_simulations}...", end='\r')

            # ======================================================
            # Stratified Sampling
            # ======================================================
            group_idx = sim % 3  

            if group_idx == 0:
                current_num_s = np.random.randint(5, 13) 
                current_num_r = np.random.randint(1, 3)  
            elif group_idx == 1:
                current_num_s = np.random.randint(13, 29)
                current_num_r = np.random.randint(2, 4)
            else:
                current_num_s = np.random.randint(29, 51)
                current_num_r = np.random.randint(4, 6)
            
            # ======================================================
            # Asymmetric Competition
            # ======================================================
            gamma_intra = float(np.random.uniform(0.95, 1.05))
            gamma_ia = gamma_intra
            gamma_jb = gamma_intra

            gamma_ja = float(np.random.uniform(0.80, 1.20))
            delta = float(np.random.uniform(DELTA_MIN, DELTA_MAX))
            gamma_ib = min(gamma_ja + delta, GAMMA_IB_CAP)
            
            if gamma_ib <= gamma_ja:
                gamma_ib = gamma_ja + EPS
            # ======================================================

            S0 = S0_total / current_num_s
            R0 = R0_total / current_num_r

            MICr = np.random.uniform(64.0, 256.0, current_num_r)
            Kr   = np.random.uniform(1.0, 2.0, current_num_r)
            current_Vrmax = np.random.uniform(Vrmax_range[0], Vrmax_range[1], current_num_r)

            if mode == 'MIC':
                current_MICs = np.random.uniform(r_min, r_max, current_num_s)
                current_Ks   = np.random.uniform(4.0, 6.0, current_num_s)
            else:
                current_MICs = np.random.uniform(2.0, 8.0, current_num_s)
                current_Ks   = np.random.uniform(r_min, r_max, current_num_s)

            current_Vsmax = np.random.uniform(Vsmax_range[0], Vsmax_range[1], current_num_s)

            SC_curve_sim = []

            for j, c in enumerate(c_range):
                S = np.full(current_num_s, S0, dtype=float)
                R = np.full(current_num_r, R0, dtype=float)

                term_s = (c / current_MICs) ** current_Ks
                Vs = current_Vsmax - ((current_Vsmax - Vsmin) * term_s) / (term_s - (Vsmin / current_Vsmax))
                term_r = (c / MICr) ** Kr
                Vr = current_Vrmax - ((current_Vrmax - Vrmin) * term_r) / (term_r - (Vrmin / current_Vrmax))

                for _ in range(time_steps):
                    total_S = np.sum(S); total_R = np.sum(R)
                    comp_s = 1 - (gamma_ia * total_S + gamma_ja * total_R) / K_cap
                    comp_r = 1 - (gamma_ib * total_S + gamma_jb * total_R) / K_cap
                    
                    S += (Vs * S * comp_s - d * S) * dt
                    R += (Vr * R * comp_r - d * R) * dt
                    
                    S[S < 0] = 0; R[R < 0] = 0

                total_S_end = np.sum(S); total_S_start = current_num_s * S0
                total_R_end = np.sum(R); total_R_start = current_num_r * R0

                if total_R_end < 1.0:
                    sc_val = 10.0  # SC_CAP
                else:
                    sc_val = (total_S_end / total_S_start) / (total_R_end / total_R_start)

                if np.isinf(sc_val) or np.isnan(sc_val) or sc_val > 10.0:
                    sc_val = 10.0
                SC_curve_sim.append(sc_val)

            SC_curve_sim = np.array(SC_curve_sim)
            all_curves_matrix.append(SC_curve_sim)

            try:
                if np.min(SC_curve_sim) < 1.0 and np.max(SC_curve_sim) > 1.0:
                    msc_val = np.interp(1.0, SC_curve_sim[::-1], c_range[::-1])
                    msc_list.append(msc_val)
                else:
                    msc_list.append(np.nan)
            except:
                msc_list.append(np.nan)

        print("     -> Done.                                        ")
        all_curves_matrix = np.array(all_curves_matrix)
        results[f"{r_min}-{r_max}"] = {
            'msc': np.array(msc_list),
            'sc_mean': np.nanmean(all_curves_matrix, axis=0),
            'sc_std': np.nanstd(all_curves_matrix, axis=0),
            'c_axis': c_range
        }
    return results

# ==========================================
# 4. Execute Calculations
# ==========================================
mic_ranges = [(2.0, 4.0), (4.0, 6.0), (6.0, 8.0)]
results_mic = run_simulation('MIC', mic_ranges, c_max=40)

k_ranges = [(4.0, 4.6), (4.6, 5.3), (5.3, 6.0)]
results_k = run_simulation('K', k_ranges, c_max=40)

# ==========================================
# 5. Export to Excel
# ==========================================
# Define output directory using a relative path for GitHub compatibility
save_dir = "./results_physiological"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

def export_results_to_excel(res_mic, res_k, directory, filename="MSC_N999_Stratified.xlsx"):
    print(f"\n--- Exporting Data to Excel ---")
    data_dict = {}
    for label, res in res_mic.items():
        data_dict[f"MICs_{label}"] = res['msc']
    for label, res in res_k.items():
        data_dict[f"Ks_{label}"] = res['msc']

    save_path = os.path.join(directory, filename)
    try:
        pd.DataFrame({k: pd.Series(v) for k, v in data_dict.items()}).to_excel(save_path, index_label="Sim_ID")
        print(f"Excel saved to: {save_path}")
    except Exception as e:
        print(f"Failed to save Excel: {e}")

export_results_to_excel(results_mic, results_k, save_dir)

# ==========================================
# 6. Plotting (Oversized Fonts and Independent System)
# ==========================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'stix'  

# --- Style Configuration (Inherited oversized parameters) ---
CONFIG = {
    'label_size': 55,        # Axis title font size
    'tick_size': 54,         # Axis tick number size
    'legend_size': 40,       # Legend font size
    'annotation_size': 45,   # "SC = 1" annotation size
    
    'border_width': 5,       # Axis border and tick mark linewidth
    'plot_linewidth': 5,     # Curve linewidth
    'ref_line_width': 6,     # SC=1 horizontal reference linewidth
    'ref_line_color': 'black', # Reference line color
    
    'mic_bins': 200, 'k_bins': 200,
    'major_tick_length': 14, # Major tick length
    
    'single_fig_size': (18, 24), # Width and height of a single split figure
    'y_label_x_pos': -0.12   # Y-axis title offset (fine-tuned to prevent overlap with oversized fonts)
}

# --- Color Configuration (Restore original blue/red palette) ---
colors_mic = ['#6BAED6', '#3182BD', '#08519C'] # Blue palette
colors_k   = ['#FB6A4A', '#CB181D', '#67000D'] # Red palette

# --- Utility Functions ---
def calculate_95ci_margin(std_dev, n):
    se = std_dev / np.sqrt(n)
    t_critical = stats.t.ppf(0.975, df=n-1)
    return t_critical * se

def apply_global_style(axes_list):
    """Apply global oversized ticks and border styles"""
    for ax in axes_list:
        ax.tick_params(axis='both', which='major',
                       labelsize=CONFIG['tick_size'],
                       width=CONFIG['border_width'],
                       length=CONFIG['major_tick_length'])
        for spine in ax.spines.values():
            spine.set_linewidth(CONFIG['border_width'])


# ----------------------------------------------------
# Plot Figure 1: MIC (Independent logic for the left side)
# ----------------------------------------------------
MIC_X_LIM = (0, 6)
fig_mic, (ax_mic_top, ax_mic_bot) = plt.subplots(
    2, 1, figsize=CONFIG['single_fig_size'], dpi=100, 
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0}
)

global_max_y_mic = 0

for idx, (label, res) in enumerate(results_mic.items()):
    mean = res['sc_mean']; std = res['sc_std']; c = res['c_axis']
    color_val = colors_mic[idx]
    
    if len(mean) > 0:
        # Note on label format: Only put formulas inside $$, keep numbers outside
        ax_mic_top.plot(c, mean, color=color_val, linewidth=CONFIG['plot_linewidth'], label=rf'$MIC_S$: {label}')
        ci_margin = calculate_95ci_margin(std, num_simulations)
        ax_mic_top.fill_between(c, np.maximum(mean - ci_margin, 0), mean + ci_margin, color=color_val, alpha=0.4, edgecolor=None)

    valid_indices = (~np.isnan(res['msc'])) & (res['msc'] != 0)
    clean_data = res['msc'][valid_indices]
    
    if len(clean_data) > 0:
        n_hist, _, _ = ax_mic_bot.hist(clean_data, bins=CONFIG['mic_bins'], range=MIC_X_LIM,
                                     color=color_val, alpha=0.5, edgecolor=color_val, linewidth=0.8)
        if len(n_hist) > 0:
            global_max_y_mic = max(global_max_y_mic, np.max(n_hist))

# Formatting MIC
ax_mic_top.axhline(1.0, color=CONFIG['ref_line_color'], linestyle='--', linewidth=CONFIG['ref_line_width'], alpha=0.8)
ax_mic_top.set_ylabel("Selection Coefficient (SC)", fontsize=CONFIG['label_size'])

# [Core modification] Force legend to use Arial font
ax_mic_top.legend(loc='upper right', frameon=False, prop={'family': 'Arial', 'size': CONFIG['legend_size']})

ax_mic_top.text(5.9, 1.25, "SC = 1", fontsize=CONFIG['annotation_size'], color=CONFIG['ref_line_color'], ha='right', weight='bold')

for ax in [ax_mic_top, ax_mic_bot]:
    ax.set_xlim(*MIC_X_LIM)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    
apply_global_style([ax_mic_top, ax_mic_bot])

ax_mic_top.set_xticklabels([])
ax_mic_bot.set_xlabel("Antibiotic concentration (c)", fontsize=CONFIG['label_size'], labelpad=15)
ax_mic_bot.set_ylabel("Frequency", fontsize=CONFIG['label_size'])
ax_mic_bot.set_ylim(0, global_max_y_mic * 1.15 if global_max_y_mic > 0 else 1)
ax_mic_bot.yaxis.set_major_locator(MaxNLocator(integer=True))

fig_mic.align_ylabels([ax_mic_top, ax_mic_bot])
ax_mic_top.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)
ax_mic_bot.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)

image_path_mic = os.path.join(save_dir, "MSC_N999_Stratified_MIC_Curves.png")
try:
    fig_mic.savefig(image_path_mic, dpi=600, bbox_inches='tight')
    print(f"Independent MIC figure saved to: {image_path_mic}")
except Exception as e:
    print(f"Failed to save MIC figure: {e}")


# ----------------------------------------------------
# Plot Figure 2: K (Independent logic for the right side)
# ----------------------------------------------------
K_X_LIM = (0, 6)
fig_k, (ax_k_top, ax_k_bot) = plt.subplots(
    2, 1, figsize=CONFIG['single_fig_size'], dpi=100, 
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0}
)

global_max_y_k = 0

for idx, (label, res) in enumerate(results_k.items()):
    mean = res['sc_mean']; std = res['sc_std']; c = res['c_axis']
    color_val = colors_k[idx]
    
    if len(mean) > 0:
        # Note on label format: Only put formulas inside $$, keep numbers outside
        ax_k_top.plot(c, mean, color=color_val, linewidth=CONFIG['plot_linewidth'], label=rf'$K_S$: {label}')
        ci_margin = calculate_95ci_margin(std, num_simulations)
        ax_k_top.fill_between(c, np.maximum(mean - ci_margin, 0), mean + ci_margin, color=color_val, alpha=0.4, edgecolor=None)

    valid_indices = (~np.isnan(res['msc'])) & (res['msc'] != 0)
    clean_data = res['msc'][valid_indices]
    
    if len(clean_data) > 0:
        n_hist, _, _ = ax_k_bot.hist(clean_data, bins=CONFIG['k_bins'], range=K_X_LIM,
                                     color=color_val, alpha=0.5, edgecolor=color_val, linewidth=0.8)
        if len(n_hist) > 0:
            global_max_y_k = max(global_max_y_k, np.max(n_hist))

# Formatting K
ax_k_top.axhline(1.0, color=CONFIG['ref_line_color'], linestyle='--', linewidth=CONFIG['ref_line_width'], alpha=0.8)
ax_k_top.set_ylabel("Selection Coefficient (SC)", fontsize=CONFIG['label_size'])

# [Core modification] Force legend to use Arial font
ax_k_top.legend(loc='upper right', frameon=False, prop={'family': 'Arial', 'size': CONFIG['legend_size']})

ax_k_top.text(5.9, 1.25, "SC = 1", fontsize=CONFIG['annotation_size'], color=CONFIG['ref_line_color'], ha='right', weight='bold')

for ax in [ax_k_top, ax_k_bot]:
    ax.set_xlim(*K_X_LIM)
    ax.xaxis.set_major_locator(MultipleLocator(1))

apply_global_style([ax_k_top, ax_k_bot])

ax_k_top.set_xticklabels([])
ax_k_bot.set_xlabel("Antibiotic concentration (c)", fontsize=CONFIG['label_size'], labelpad=15)
ax_k_bot.set_ylabel("Frequency", fontsize=CONFIG['label_size'])
ax_k_bot.set_ylim(0, global_max_y_k * 1.15 if global_max_y_k > 0 else 1)
ax_k_bot.yaxis.set_major_locator(MaxNLocator(integer=True))

fig_k.align_ylabels([ax_k_top, ax_k_bot])
ax_k_top.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)
ax_k_bot.yaxis.set_label_coords(CONFIG['y_label_x_pos'], 0.5)

image_path_k = os.path.join(save_dir, "MSC_N999_Stratified_K_Curves.png")
try:
    fig_k.savefig(image_path_k, dpi=600, bbox_inches='tight')
    print(f"Independent Ks figure saved to: {image_path_k}")
except Exception as e:
    print(f"Failed to save Ks figure: {e}")

# ==========================================
# 7. Display Plots
# ==========================================
plt.show()